In [35]:
import pandas as pd
import numpy as np
import string


INPUT_XLS  = r'F:/modelling_allographic_letter_recognition/English/words_with_vectors.xlsx'        
OUTPUT_XLS = r'F:/modelling_allographic_letter_recognition/English/SL_DF_Case.xlsx'    
MAX_LEN    = 10                                



ALPHABET52 =  list(string.ascii_lowercase) + list(string.ascii_uppercase)
IDX = {ch: i for i, ch in enumerate(ALPHABET52)}


def identity_vector(word: str) -> list[int]:
    v = [0] * 52
    for ch in word:
        i = IDX.get(ch)
        if i is not None:
            v[i] += 1
    return v


def position_vector(word: str, m: int = MAX_LEN) -> list[int]:
    letters = [ch for ch in word if ch in IDX]
    if len(letters) > m:
        raise ValueError(f'"{word}" longer than MAX_LEN={m}')
    slots = sorted(set(letters), key=IDX.get)      # distinct letters, A-Z-a-z order
    if len(slots) > m:
        raise ValueError(f'"{word}" has {len(slots)} distinct letters > MAX_LEN')

    grid = np.zeros((m, m), dtype=int)
    for pos, ch in enumerate(letters, start=1):    # 1-based character position
        grid[slots.index(ch), pos - 1] = 1
    return grid.ravel().tolist()                   # length = m² (here 100)

# build encodings
df_in   = pd.read_excel(INPUT_XLS, engine='openpyxl')
id_rows = []
pos_rows = []

for w in df_in['SL_DF_Case'].astype(str):
    id_vec  = identity_vector(w)
    pos_vec = position_vector(w)

    id_rows.append({'word': w, **{f'id_{i:02d}': v  for i, v in enumerate(id_vec)}})
    pos_rows.append({'word': w, **{f'pos_{i:03d}': v for i, v in enumerate(pos_vec)}})

df_id  = pd.DataFrame(id_rows)
df_pos = pd.DataFrame(pos_rows)

# write two sheets
with pd.ExcelWriter(OUTPUT_XLS, engine='openpyxl') as xls:
    df_id.to_excel(xls, sheet_name='identity', index=False)
    df_pos.to_excel(xls, sheet_name='position', index=False)


In [13]:
ALPHABET52 = list('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ')
ALPHA2IDX  = {ch: i for i, ch in enumerate(ALPHABET52)}
MAX_LEN    = 10              # adjust if needed (≥ longest word)

def orthography_vector(word: str) -> list[int]:
    vec = [0]*(52*MAX_LEN)
    for pos, ch in enumerate(word, start=0):   # zero-based position
        if pos >= MAX_LEN:                     # ignore over-long words
            break
        idx = ALPHA2IDX.get(ch)
        if idx is not None:
            vec[pos*52 + idx] = 1             # column-major flatten
    return vec


In [16]:
len(orthography_vector("Same"))

520

In [20]:
INPUT_XLS  = r'F:/modelling_allographic_letter_recognition/English/words_with_vectors.xlsx'        
OUTPUT_XLS = r'F:/modelling_allographic_letter_recognition/English/orthograph_encodings.xlsx'    


ALPHABET52 = list('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ')
ALPHA2IDX  = {ch: i for i, ch in enumerate(ALPHABET52)}
MAX_LEN    = 10              

def orthography_vector(word: str) -> list[int]:
    vec = [0]*(52*MAX_LEN)
    for pos, ch in enumerate(word, start=0):   # zero-based position
        if pos >= MAX_LEN:                     # ignore over-long words
            break
        idx = ALPHA2IDX.get(ch)
        if idx is not None:
            vec[pos*52 + idx] = 1             # column-major flatten
    return vec

df_in   = pd.read_excel(INPUT_XLS, engine='openpyxl')

orth_rows = []

for w in df_in['word'].astype(str):
    orth_vec  = orthography_vector(w)
    orth_rows.append({'word': w, **{f'v_{i:02d}': v  for i, v in enumerate(orth_vec)}})

df_orth  = pd.DataFrame(orth_rows)


with pd.ExcelWriter(OUTPUT_XLS, engine='openpyxl') as xls:
    df_orth.to_excel(xls, sheet_name='orth_vector', index=False)

## Non-Case-Sensitive Words 

Here we treat upper and lower case letters the same. 

In [28]:
# Orthographic encoding 

import pandas as pd
import numpy as np
import string

INPUT_XLS  = r'F:/modelling_allographic_letter_recognition/English/words_with_vectors.xlsx'        
OUTPUT_XLS = r'F:/modelling_allographic_letter_recognition/English/Non_case_sensitive_data/Non_case_Sensitive_orthograph_encodings.xlsx'    


ALPHABET26 = list('abcdefghijklmnopqrstuvwxyz')
ALPHA2IDX  = {ch: i for i, ch in enumerate(ALPHABET26)}
MAX_LEN    = 10              

def orthography_vector(word: str) -> list[int]:
    vec = [0]*(26*MAX_LEN)
    for pos, ch in enumerate(word, start=0):   # zero-based position
        if pos >= MAX_LEN:                     # ignore over-long words
            break
        idx = ALPHA2IDX.get(ch)
        if idx is not None:
            vec[pos*26 + idx] = 1             
    return vec

df_in   = pd.read_excel(INPUT_XLS, engine='openpyxl')

orth_rows = []

for w in df_in['word'].astype(str):
    orth_vec  = orthography_vector(w.lower())
    orth_rows.append({'word': w, **{f'v_{i:02d}': v  for i, v in enumerate(orth_vec)}})

df_orth  = pd.DataFrame(orth_rows)


with pd.ExcelWriter(OUTPUT_XLS, engine='openpyxl') as xls:
    df_orth.to_excel(xls, sheet_name='orth_vector', index=False)

In [30]:
#t-test data encoding

import pandas as pd
import numpy as np
import string


INPUT_XLS  = r'F:/modelling_allographic_letter_recognition/English/words_with_vectors.xlsx'        # source (contains column "word")
OUTPUT_XLS = r'F:/modelling_allographic_letter_recognition/English/Non_case_sensitive_data/Non_case_Sensitive_word_encodings.xlsx'    # destination
MAX_LEN    = 10                               


# 26-letter alphabet: 0-25 = 'a'-'z'
ALPHABET26 =  list(string.ascii_lowercase) 
IDX = {ch: i for i, ch in enumerate(ALPHABET26)}


def identity_vector(word: str) -> list[int]:
    v = [0] * 26
    for ch in word.lower():
        i = IDX.get(ch)
        if i is not None:
            v[i] += 1
    return v


def position_vector(word: str, m: int = MAX_LEN) -> list[int]:
    letters = [ch for ch in word.lower() if ch in IDX]
    if len(letters) > m:
        raise ValueError(f'"{word}" longer than MAX_LEN={m}')
    slots = sorted(set(letters), key=IDX.get)      # distinct letters, A-Z-a-z order
    if len(slots) > m:
        raise ValueError(f'"{word}" has {len(slots)} distinct letters > MAX_LEN')

    grid = np.zeros((m, m), dtype=int)
    for pos, ch in enumerate(letters, start=1):    # 1-based character position
        grid[slots.index(ch), pos - 1] = 1
    return grid.ravel().tolist()                   # length = m² (here 100)


df_in   = pd.read_excel(INPUT_XLS, engine='openpyxl')
id_rows = []
pos_rows = []

for w in df_in['word'].astype(str):
    id_vec  = identity_vector(w)
    pos_vec = position_vector(w)

    id_rows.append({'word': w, **{f'id_{i:02d}': v  for i, v in enumerate(id_vec)}})
    pos_rows.append({'word': w, **{f'pos_{i:03d}': v for i, v in enumerate(pos_vec)}})

df_id  = pd.DataFrame(id_rows)
df_pos = pd.DataFrame(pos_rows)


with pd.ExcelWriter(OUTPUT_XLS, engine='openpyxl') as xls:
    df_id.to_excel(xls, sheet_name='identity', index=False)
    df_pos.to_excel(xls, sheet_name='position', index=False)


In [10]:
ALPHABET26 =  list(string.ascii_lowercase) 
IDX = {ch: i for i, ch in enumerate(ALPHABET26)}
MAX_LEN    = 10

def position_vector(word: str, m: int = MAX_LEN) -> list[int]:
    letters = [ch for ch in word.lower() if ch in IDX]
    if len(letters) > m:
        raise ValueError(f'"{word}" longer than MAX_LEN={m}')
    slots = sorted(set(letters), key=IDX.get)      # distinct letters, A-Z-a-z order
    if len(slots) > m:
        raise ValueError(f'"{word}" has {len(slots)} distinct letters > MAX_LEN')

    grid = np.zeros((m, m), dtype=int)
    for pos, ch in enumerate(letters, start=1):    # 1-based character position
        grid[slots.index(ch), pos - 1] = 1
    vect= grid.ravel().tolist()
    return vect


In [12]:
position_vector("government")

[0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]